# Parameter Sensitivity Explorer

This notebook performs a **vectorized grid search** using the `ggTrader` orchestrator api. It visualizes the profitability landscape to find robust parameter regions.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import vectorbt as vbt
import plotly.graph_objects as go
from tabulate import tabulate

# Auto-reload custom modules
%load_ext autoreload
%autoreload 2

# Ensure project root is in path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..',  'src'))
if project_root not in sys.path:
    sys.path.append(project_root)

from ggTrader.core.orchestrator import run_sensitivity_orchestrator

print("Environment initialized.")

Environment initialized.


In [ ]:
# --- Configuration ---
CONSTANTS = {
    "SYMBOLS": ["BTC"],
    "SYMBOLS_FILE": os.path.join(os.getcwd(), "..", "data", "top_20_USD_2025-01-01_2025-12-31_movers.json"),
    "START_DATE": "2023-01-01",
    "END_DATE": "2025-12-31",
    "INTERVAL": "4h",
    "START_CASH": 1000,
    "PORTFOLIO_SHARE": 1,
    "FEES": 0.004, # kraken max 0.4%
    "MIN_TRADES": 5,
    "SLIPPAGE": 0.003,
}

print("Configuration loaded.")

Configuration loaded.


In [ ]:
# --- Define Parameter Grid ---
params = {
    # entry
    "sar_acceleration": [0.02],
    "sar_maximum": [0.2],
    "use_dmp_cross": [False],
    "adx_threshold": list(range(25, 46, 5)), 
    "adx_length": list(range(15, 36, 5)), 
    # exit
    "atr_length": list(range(5, 21, 5)), #  varies a lot
    "atr_multiplier": list(np.arange(0.1, 1.1, 0.1)),  # small values work for some reason

}

print("Parameter grid defined.")

Parameter grid defined.


In [ ]:
# --- Run Vectorized Analysis ---
# Note: show_progress=True enables VectorBT's tqdm progress bar
results = run_sensitivity_orchestrator(
    config=CONSTANTS, param_grid=params, save_results=False, show_progress=True
)

results_df = results["results_df"]
best_params = results["best_params"]

print("\nAnalysis Complete.")

Loading data...
Running Vectorized Sensitivity Analysis in 2 chunks (600 total combinations, chunk_size=500)...
  > Processing chunk 1 of 2 (0 to 500)...


  0%|          | 0/500 [00:00<?, ?it/s]

  > Processing chunk 2 of 2 (500 to 600)...


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]


Analysis Complete.


In [ ]:
top_n = 10

top = results_df.sort_values("Sharpe Ratio", ascending=False).head(top_n)
stats = top.agg(['mean', 'std', 'min', 'max']).round(3)
print("\nTOP 10 COMBINATIONS:")
print(tabulate(top.head(10).round(2), headers="keys", tablefmt="simple", showindex=False))

print("\n TOP 10 PARAMETER STATS")
print(tabulate(stats.T, headers='keys', tablefmt='simple', numalign="right"))



TOP 10 COMBINATIONS:
  adx_length    adx_threshold    sar_acceleration    sar_maximum  use_dmp_cross      atr_length    atr_multiplier    Sharpe Ratio
------------  ---------------  ------------------  -------------  ---------------  ------------  ----------------  --------------
          20               40                0.02            0.2  False                       5               0.8            2.21
          20               40                0.02            0.2  False                      10               0.7            2.17
          20               40                0.02            0.2  False                      10               0.5            2.04
          20               40                0.02            0.2  False                      10               0.6            2.04
          20               40                0.02            0.2  False                      20               0.5            2.02
          20               40                0.02            0.2  Fa

In [ ]:
# --- Visualization Helper ---
def show_heatmap(df, x_param, y_param, metric="Sharpe Ratio"):
    """Generates and displays a heatmap for the given parameter pair."""
    heatmap_data = df.pivot_table(
        index=y_param, 
        columns=x_param, 
        values=metric,
        aggfunc="mean"
    )

    fig = go.Figure(data=go.Heatmap(
        z=heatmap_data.values,
        x=heatmap_data.columns,
        y=heatmap_data.index,
        colorscale='Viridis',
        colorbar=dict(title=metric)
    ))

    fig.update_layout(
        title=f"{metric} Landscape: {y_param} vs {x_param}",
        xaxis_title=x_param,
        yaxis_title=y_param
    )

    fig.show()

print("Visualization helper defined.")

Visualization helper defined.


In [ ]:
# --- Generate Heatmaps ---
# You can list any pairs of parameters you want to explore
pairs_to_plot = [
    ("adx_threshold", "adx_length"), #entry
    # ("sar_acceleration", "sar_maximum"),
    ("atr_multiplier", "atr_length"), #exit
    ("adx_length", "atr_length"),
    # ("use_dmp_cross", "adx_length"),
    # ("use_dmp_cross", "adx_threshold")
]

for x, y in pairs_to_plot:
    show_heatmap(results_df, x, y)

## Single Backtest Run with Best Parameters

Running a full backtest with the best parameters found to generate detailed statistics and plots.

In [ ]:
from ggTrader.core.orchestrator import run_backtest_orchestrator

# --- Run Single Backtest with Best Parameters ---
print(f"Running single backtest with best parameters: {best_params}")
# Option B: Print directly as a list of items
print(tabulate(best_params.items(), headers=['Parameter', 'Value'], tablefmt='github'))
backtest_res = run_backtest_orchestrator(
    config=CONSTANTS, 
    params=best_params, 
    save_results=False, 
    show_progress=True
)



Running single backtest with best parameters: {'adx_length': 20, 'adx_threshold': 40, 'sar_acceleration': 0.02, 'sar_maximum': 0.2, 'use_dmp_cross': False, 'atr_length': 5, 'atr_multiplier': 0.8}
| Parameter        |   Value |
|------------------|---------|
| adx_length       |   20    |
| adx_threshold    |   40    |
| sar_acceleration |    0.02 |
| sar_maximum      |    0.2  |
| use_dmp_cross    |    0    |
| atr_length       |    5    |
| atr_multiplier   |    0.8  |
Loading data...
Running backtest...


  0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
pf = backtest_res["portfolio"]
stats = backtest_res["stats"]

print("\n--- Backtest Statistics ---\n")


# --- Visualization ---
print("Global Portfolio Stats:")
# Convert stats to a DataFrame for a cleaner table view
stats_df = pf.stats().to_frame(name="Value").reset_index()

# 2. Format only the numbers to 2 decimals (ignoring dates and durations)
def format_values(x):
    if isinstance(x, (float, np.floating)):
        return f"{x:.2f}"
    return str(x)
stats_df["Value"] = stats_df["Value"].apply(format_values)

# 3. Print with tabulate (no floatfmt needed now because we formatted them in step 2)
print(tabulate(stats_df, headers=["Metric", "Value"], tablefmt="simple", numalign="right", showindex=False))


--- Backtest Statistics ---

Global Portfolio Stats:
Metric                      Value
--------------------------  --------------------------
Start                       2023-01-01 00:00:00+00:00
End                         2025-12-30 16:00:00+00:00
Period                      1094 days 20:00:00
Start Value                 1000.00
End Value                   5542.91
Total Return [%]            454.29
Benchmark Return [%]        433.46
Max Gross Exposure [%]      100.00
Total Fees Paid             441.37
Max Drawdown [%]            17.93
Max Drawdown Duration       239 days 20:00:00
Total Trades                18
Total Closed Trades         17
Total Open Trades           1
Open Trade PnL              180.28
Win Rate [%]                64.71
Best Trade [%]              48.03
Worst Trade [%]             -4.04
Avg Winning Trade [%]       18.72
Avg Losing Trade [%]        -1.68
Avg Winning Trade Duration  26 days 07:16:21.818181818
Avg Losing Trade Duration   2 days 12:00:00
Profit Factor 

In [ ]:
pf.plot(subplots=['drawdowns','value','cum_returns' ]).show()

In [ ]:
print(pf.wrapper.columns)

MultiIndex([(20, 40, 0.02, 0.2, False, 5, 0.8, 'BTC')],
           names=['sf_adx_length', 'sf_adx_threshold', 'sf_sar_acceleration', 'sf_sar_maximum', 'sf_use_dmp_cross', 'sf_atr_length', 'sf_atr_multiplier', 'symbol'])


In [ ]:
# 1. Run backtest for just BTC
btc_pf = run_backtest_orchestrator(
    config={**CONSTANTS, "SYMBOLS": ["BTC"],"PORTFOLIO_SHARE": 1,"USE_CASH_SHARING": False,"group_by": False}, 
    params=best_params, 
    save_results=False
)['portfolio']
# 2. Plot including PnL subplots
from tabulate import tabulate

# --- Visualization ---
print("Global Portfolio Stats:")
# Convert stats to a DataFrame for a cleaner table view
stats_df = btc_pf.stats().to_frame(name="Value").reset_index()

# 2. Format only the numbers to 2 decimals (ignoring dates and durations)
def format_values(x):
    if isinstance(x, (float, np.floating)):
        return f"{x:.2f}"
    return str(x)
stats_df["Value"] = stats_df["Value"].apply(format_values)

# 3. Print with tabulate (no floatfmt needed now because we formatted them in step 2)
print(tabulate(stats_df, headers=["Metric", "Value"], tablefmt="simple", numalign="right", showindex=False))



Loading data...
Running backtest...
Global Portfolio Stats:


C:\Users\gkuep\AppData\Local\Temp\ipykernel_2228\2438061421.py:13: UserWarning:

Metric 'profit_factor' raised an exception



ValueError: assignment destination is read-only

In [ ]:
btc_pf["portfolio"].iloc[0].plot().show()
